In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)


In [6]:
map = geemap.Map()

point = ee.Geometry.Point([90.4152, 23.8041]) #around dhaka
region = ee.Geometry.Rectangle([90.3, 23.7, 90.5, 23.9]) #bounding box around the point

map.centerObject(point, 10)
map.addLayer(point, {'color': 'red'}, 'Point Layer')
map.addLayer(region, {'color': 'blue'}, 'Region Layer')

map #just testing if everything is working

Map(center=[23.8041, 90.4152], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [7]:
#collecting data
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") #Sentinel-2 Surface Reflectance data

filtered_image = s2 \
    .filterBounds(region) \
    .filterDate('2022-01-01', '2022-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

print(f"Number of images found: {filtered_image.size().getInfo()}")

Number of images found: 36


In [9]:
#reducing to a single image by taking median

median_image = filtered_image.median()

In [14]:
# Create a new map for the median image
map2 = geemap.Map()

vis_params = {
    'bands': ['B4', 'B3', 'B2'],  # RGB
    'min': 0,
    'max': 3000,
    'gamma': 1.4
}

# Clip the image to only show the rectangle region
clipped_image = median_image.clip(region)

map2.centerObject(region, 10)
map2.addLayer(clipped_image, vis_params, "Median Image")

map2

Map(center=[23.800006561617693, 90.39999999999813], controls=(WidgetControl(options=['position', 'transparent_…